# ChefBot AI — Dallo scatto alla ricetta
### Progetto Finale Epicode

**Obiettivo:** costruire un sistema intelligente che:
1. Riconosce il tipo di piatto da una foto (Computer Vision / Transfer Learning)
2. Arricchisce la predizione con dati strutturati: ingredienti, calorie, descrizione (Knowledge Retrieval)
3. Permette di cercare un piatto descrivendo un "umore" o un desiderio, tramite ricerca semantica su embeddings (NLP / Semantic Search)

**Dataset:** [Food101], useremo un subset di 15 classi

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import tensorflow as tf
import tensorflow_datasets as tfds
from keras import layers, models

# Riproducibilità
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

---
## 1. Riconoscimento Visivo (Computer Vision)

Carichiamo il dataset **Food101** tramite `tensorflow_datasets`, selezioniamo un subset di classi e costruiamo un
classificatore basato su **Transfer Learning** con **MobileNetV2** pre-addestrata su ImageNet.

### 1.1 Caricamento del dataset e selezione delle classi

In [ ]:
# Carichiamo il dataset Food101 (train e validation)
(ds_train_full, ds_test_full), ds_info = tfds.load(
    'food101',
    split=['train', 'validation'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True
)

all_class_names = ds_info.features['label'].names
print(f"Numero totale di classi nel dataset: {len(all_class_names)}")
print("Esempio di classi:", all_class_names[:10])

In [ ]:
# Selezioniamo un subset di 15 classi
SELECTED_CLASSES = [
    'baklava', 'churros', 'pizza', 'sushi', 'hamburger',
    'caesar_salad', 'tiramisu', 'ramen', 'french_fries', 'tacos',
    'ice_cream', 'lasagna', 'omelette', 'risotto', 'waffles'
]

# Mappiamo gli indici originali del dataset (0-100) verso i nostri nuovi indici (0-14)
selected_indices = [all_class_names.index(c) for c in SELECTED_CLASSES]
old_to_new_label = {old: new for new, old in enumerate(selected_indices)}

NUM_CLASSES = len(SELECTED_CLASSES)
print(f"Classi selezionate ({NUM_CLASSES}):", SELECTED_CLASSES)

In [ ]:
def is_selected_class(image, label):
    """Filtra solo gli esempi appartenenti alle classi scelte."""
    return tf.reduce_any(tf.equal(label, selected_indices))

_remap_array = np.full(len(all_class_names), -1, dtype=np.int64)
for new_idx, old_idx in enumerate(selected_indices):
    _remap_array[old_idx] = new_idx
_remap_tensor = tf.constant(_remap_array, dtype=tf.int64)

def remap_label(image, label):
    """Rimappa l'etichetta originale (0-100) al nuovo indice (0-14) tramite tf.gather"""
    new_label = tf.gather(_remap_tensor, tf.cast(label, tf.int64))
    return image, new_label

ds_train = ds_train_full.filter(is_selected_class).map(remap_label, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test_full.filter(is_selected_class).map(remap_label, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
# Test: controlliamo che nessuna etichetta rimappata sia -1
sample_labels = [int(l.numpy()) for _, l in ds_train.take(200)]
assert min(sample_labels) >= 0, "Trovate etichette -1! Controllare il remapping."
print("Range etichette nel campione:", min(sample_labels), "-", max(sample_labels))
print("Sanity check superato ✅")

### 1.2 Preprocessing e Data Pipeline

Ridimensioniamo le immagini alla dimensione richiesta da MobileNetV2 (224x224) e applichiamo la normalizzazione
specifica del modello. Aggiungiamo anche semplice data augmentation sul training set.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

ds_train_preprocessed = (ds_train
                          .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
                          .cache('/content/train_cache'))
ds_test_preprocessed = (ds_test
                         .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
                         .cache('/content/test_cache'))

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def augment(image, label):
    image = data_augmentation(image, training=True)
    return image, label

train_ds = (ds_train_preprocessed
            .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
            .shuffle(1000)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))

test_ds = (ds_test_preprocessed
           .batch(BATCH_SIZE)
           .prefetch(tf.data.AUTOTUNE))

In [ ]:
# Visualizziamo qualche esempio dal training set per verifica
class_names_map = {i: c for i, c in enumerate(SELECTED_CLASSES)}

plt.figure(figsize=(12, 8))
for i, (image, label) in enumerate(ds_train.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image.numpy().astype("uint8"))
    label_idx = int(label.numpy())
    plt.title(class_names_map[label_idx])
    plt.axis("off")
plt.tight_layout()
plt.show()

### 1.3 Costruzione del modello (Transfer Learning con MobileNetV2)

Usiamo **MobileNetV2** pre-addestrata su ImageNet come *feature extractor*, congelando i suoi pesi (`base_model.trainable = False`)
e aggiungendo un classificatore finale personalizzato per le nostre 15 classi.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # congeliamo i pesi pre-addestrati

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs, name="ChefBot_Vision_Classifier")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')]
)

model.summary()

### 1.4 Training

Alleniamo solo il classificatore finale (i pesi della base restano congelati).

In [ ]:
EPOCHS = 10

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=3, restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    callbacks=[early_stop]
)

### 1.5 Fine-tuning

Una volta stabilizzato il classificatore, possiamo scongelare gli ultimi layer della base per un fine-tuning più fine,
usando un learning rate molto basso per non distruggere i pesi pre-addestrati.

In [ ]:
base_model.trainable = True

# Congeliamo tutti i layer tranne gli ultimi 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')]
)

fine_tune_epochs = 5
history_fine = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=fine_tune_epochs,
    callbacks=[early_stop]
)

### 1.6 Valutazione del modello

In [ ]:
results = model.evaluate(test_ds)
test_loss, test_acc, test_top3 = results
print(f"Accuratezza (top-1) sul test set: {test_acc:.2%}")
print(f"Accuratezza (top-3) sul test set: {test_top3:.2%}")

In [ ]:
# Grafici di accuracy e loss
acc = history.history['accuracy'] + history_fine.history.get('accuracy', [])
val_acc = history.history['val_accuracy'] + history_fine.history.get('val_accuracy', [])
loss = history.history['loss'] + history_fine.history.get('loss', [])
val_loss = history.history['val_loss'] + history_fine.history.get('val_loss', [])

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.legend()
plt.title('Loss')
plt.tight_layout()
plt.show()

### 1.7 Matrice di confusione

Ci aiuta a capire se l'accuracy bassa è distribuita uniformemente tra le classi o se il modello confonde
sistematicamente alcune coppie di piatti (es. piatti visivamente simili).

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=SELECTED_CLASSES, yticklabels=SELECTED_CLASSES)
plt.xlabel('Predetto')
plt.ylabel('Reale')
plt.title('Matrice di confusione')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=SELECTED_CLASSES))

In [ ]:
def predict_dish(image):
    """Prende un'immagine e restituisce
    la classe predetta con la relativa confidenza."""
    img = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    img = tf.expand_dims(img, axis=0)
    preds = model.predict(img, verbose=0)[0]
    class_idx = int(np.argmax(preds))
    confidence = float(preds[class_idx])
    return class_names_map[class_idx], confidence

# Test su un'immagine del test set
for images, labels in ds_test.batch(1).take(1):
    predicted_class, conf = predict_dish(images[0].numpy())
    true_class = class_names_map[int(labels[0].numpy())]
    print(f"Classe vera: {true_class} | Predetta: {predicted_class} ({conf:.1%})")

---
## 2. Knowledge Retrieval (NLP)

Il dataset Food101 fornisce solo immagini ed etichette. Costruiamo quindi un **"dizionario della conoscenza"**
in formato JSON che mappa ogni piatto a dati strutturati: ingredienti principali, calorie stimate e una breve descrizione.

### 2.1 Costruzione del database di conoscenza

In [ ]:
knowledge_base = {
    "baklava": {
        "ingredienti": ["pasta fillo", "noci", "miele", "burro", "cannella"],
        "calorie_stimate": 330,
        "descrizione": "Dolce mediorientale a strati croccanti di pasta fillo, farcito con frutta secca e imbevuto di sciroppo di miele."
    },
    "churros": {
        "ingredienti": ["farina", "acqua", "zucchero", "cannella", "olio per friggere"],
        "calorie_stimate": 280,
        "descrizione": "Bastoncini di pasta fritta, croccanti fuori e morbidi dentro, spolverati di zucchero e cannella."
    },
    "pizza": {
        "ingredienti": ["farina", "pomodoro", "mozzarella", "olio d'oliva", "basilico"],
        "calorie_stimate": 266,
        "descrizione": "Piatto italiano a base di impasto lievitato, cotto al forno con condimenti a piacere."
    },
    "sushi": {
        "ingredienti": ["riso", "alga nori", "pesce crudo", "aceto di riso", "wasabi"],
        "calorie_stimate": 150,
        "descrizione": "Piatto giapponese a base di riso vinegrato accompagnato da pesce crudo o verdure."
    },
    "hamburger": {
        "ingredienti": ["pane", "carne di manzo", "formaggio", "lattuga", "pomodoro"],
        "calorie_stimate": 295,
        "descrizione": "Panino farcito con una polpetta di carne alla griglia e condimenti vari."
    },
    "caesar_salad": {
        "ingredienti": ["lattuga romana", "pollo grigliato", "crostini", "parmigiano", "salsa caesar"],
        "calorie_stimate": 190,
        "descrizione": "Insalata fresca e leggera con pollo, crostini croccanti e una cremosa salsa a base di acciughe."
    },
    "tiramisu": {
        "ingredienti": ["savoiardi", "mascarpone", "caffè", "cacao", "uova"],
        "calorie_stimate": 320,
        "descrizione": "Dolce al cucchiaio italiano, morbido e cremoso, con un intenso aroma di caffè e cacao."
    },
    "ramen": {
        "ingredienti": ["noodles", "brodo di maiale o pollo", "uovo", "alga nori", "cipollotto"],
        "calorie_stimate": 436,
        "descrizione": "Zuppa giapponese di noodles in brodo saporito, arricchita con uovo e topping vari."
    },
    "french_fries": {
        "ingredienti": ["patate", "olio per friggere", "sale"],
        "calorie_stimate": 312,
        "descrizione": "Bastoncini di patate fritte, croccanti fuori e morbidi dentro, un classico snack leggero."
    },
    "tacos": {
        "ingredienti": ["tortilla di mais", "carne", "cipolla", "coriandolo", "lime"],
        "calorie_stimate": 226,
        "descrizione": "Piatto messicano a base di tortilla ripiena di carne e verdure fresche."
    },
    "ice_cream": {
        "ingredienti": ["latte", "panna", "zucchero", "aromi naturali"],
        "calorie_stimate": 207,
        "descrizione": "Dessert freddo e cremoso, fresco e leggero, disponibile in moltissimi gusti."
    },
    "lasagna": {
        "ingredienti": ["pasta sfoglia", "ragù", "besciamella", "parmigiano"],
        "calorie_stimate": 310,
        "descrizione": "Piatto italiano a strati di pasta, ragù e besciamella, cotto al forno."
    },
    "omelette": {
        "ingredienti": ["uova", "sale", "pepe", "burro"],
        "calorie_stimate": 154,
        "descrizione": "Frittata soffice a base di uova sbattute e cotte in padella, semplice e proteica."
    },
    "risotto": {
        "ingredienti": ["riso carnaroli", "brodo", "burro", "parmigiano", "vino bianco"],
        "calorie_stimate": 280,
        "descrizione": "Piatto italiano cremoso a base di riso mantecato lentamente con brodo e formaggio."
    },
    "waffles": {
        "ingredienti": ["farina", "uova", "latte", "zucchero", "burro"],
        "calorie_stimate": 291,
        "descrizione": "Dolce soffice e dorato dalla caratteristica forma a griglia, spesso servito con sciroppo d'acero."
    }
}

# Salviamo il dizionario in un file JSON
with open('food_knowledge_base.json', 'w', encoding='utf-8') as f:
    json.dump(knowledge_base, f, ensure_ascii=False, indent=2)

print(f"Knowledge base creata con {len(knowledge_base)} piatti.")

### 2.2 Funzione di lookup

Data la predizione del modello di visione, recuperiamo i metadati corrispondenti dalla knowledge base.

In [ ]:
def get_dish_info(dish_name, kb=knowledge_base):
    """Recupera la scheda informativa di un piatto dato il suo nome."""
    info = kb.get(dish_name)
    if info is None:
        return None
    return {
        "piatto": dish_name.replace('_', ' ').title(),
        "ingredienti": info["ingredienti"],
        "calorie_stimate": info["calorie_stimate"],
        "descrizione": info["descrizione"]
    }

def print_dish_card(dish_name):
    """Stampa una scheda informativa leggibile."""
    info = get_dish_info(dish_name)
    if info is None:
        print(f"Nessuna informazione disponibile per '{dish_name}'.")
        return
    print(f"  {info['piatto']}")
    print(f"  {info['descrizione']}")
    print(f"  Ingredienti principali: {', '.join(info['ingredienti'])}")
    print(f"  Calorie stimate: {info['calorie_stimate']} kcal")

# Test
print_dish_card("tiramisu")

---
## 3. Ricerca Semantica tra i Piatti (Embeddings)

Implementiamo la funzione **"Ricerca per Umore"**: l'utente scrive una frase libera (es. *"voglio qualcosa di fresco e leggero"*)
e il sistema propone il piatto più vicino **semanticamente**, calcolando gli **embeddings** delle descrizioni e usando la
**similarità coseno**, anche se le parole esatte non compaiono nel testo.

### 3.1 Generazione degli embeddings per ogni piatto

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Modello multilingue leggero, adatto anche all'italiano
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
# Costruiamo un "documento semantico" per ogni piatto combinando descrizione e ingredienti
dish_names = list(knowledge_base.keys())
dish_documents = [
    f"{knowledge_base[d]['descrizione']} Ingredienti: {', '.join(knowledge_base[d]['ingredienti'])}."
    for d in dish_names
]

# Calcoliamo gli embeddings di tutti i piatti una sola volta
dish_embeddings = embedder.encode(dish_documents, convert_to_numpy=True)
print("Shape degli embeddings:", dish_embeddings.shape)

### 3.2 Funzione di ricerca semantica

In [ ]:
def search_by_mood(query, top_k=3):
    """Data una frase libera dell'utente, restituisce i piatti più vicini
    semanticamente, ordinati per similarità"""
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    similarities = cosine_similarity(query_embedding, dish_embeddings)[0]

    ranked_indices = np.argsort(similarities)[::-1][:top_k]
    results = []
    for idx in ranked_indices:
        results.append({
            "piatto": dish_names[idx].replace('_', ' ').title(),
            "similarita": float(similarities[idx]),
            "descrizione": knowledge_base[dish_names[idx]]["descrizione"]
        })
    return results

def print_search_results(query, top_k=3):
    print(f'🔍 Ricerca per: "{query}"\n')
    results = search_by_mood(query, top_k=top_k)
    for r in results:
        print(f"  → {r['piatto']}  (similarità: {r['similarita']:.2f})")
        print(f"     {r['descrizione']}\n")

In [ ]:
# Test della ricerca semantica
print_search_results("voglio qualcosa di fresco e leggero")

In [ ]:
print_search_results("dolce con miele")

In [ ]:
print_search_results("qualcosa di caldo e confortante")

---
## 4. ChefBot AI — Pipeline completa

Uniamo le tre componenti: dato uno scatto (immagine), il sistema:
1. **Riconosce** il piatto (Computer Vision)
2. **Recupera** la scheda informativa (Knowledge Retrieval)
3. Permette in qualsiasi momento una **ricerca semantica per umore** tra tutti i piatti conosciuti

In [ ]:
def chefbot_analyze(image):
    """Pipeline end-to-end: immagine -> predizione -> scheda informativa."""
    dish_name, confidence = predict_dish(image)
    print(f"📸 Piatto riconosciuto: {dish_name.replace('_', ' ').title()} (confidenza: {confidence:.1%})\n")
    print_dish_card(dish_name)
    return dish_name

# Esempio d'uso end-to-end su un'immagine del test set
for images, labels in ds_test.batch(1).take(1):
    recognized_dish = chefbot_analyze(images[0].numpy())

In [ ]:
# Esempio d'uso della ricerca per umore
print_search_results("voglio qualcosa di sfizioso da mangiare con le mani")

In [ ]:
# Esempio con foto caricata da URL
import requests
from PIL import Image
from io import BytesIO

def load_image_from_url(url):
    """Scarica un'immagine da un URL"""
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content)).convert('RGB')
    return np.array(img)


url = "https://www.giallozafferano.it/images/178-17884/Waffle_450x300.jpg"
image_from_url = load_image_from_url(url)

plt.figure(figsize=(5, 5))
plt.imshow(image_from_url)
plt.axis('off')
plt.title("Immagine caricata da URL")
plt.show()

recognized_dish = chefbot_analyze(image_from_url)